# Cross-Template Key-Value Field Extraction — Kaggle Run 3

**IE 643 Course Project** · Upgraded Architecture Training (Decoupled Curriculum, Dynamic Per-Phase Early Stopping, Focal Loss, 12-D Spatial Geometry, Soft Marginal Masking)

### Core Architectural Enhancements in Run 3:
1. **Dynamic Decoupled 3-Stage Curriculum with Early Stopping (Patience = 3):**
   - **Phase 1 (up to 10 eps):** Pure Token Classification warmup ($\lambda_{\text{link}} = 0$) ensuring stable contextual embeddings before relation learning. **Validated strictly on Entity F1** (no relation head compute).
   - **Phase 2 (up to 8 eps):** Teacher-Forced Relation Head warmup (LiLT encoder backbone frozen) to learn clean layout geometry. **Validated strictly on Linking F1 under Teacher Forcing**.
   - **Phase 3 (up to 10 eps):** Joint end-to-end fine-tuning with differential learning rates. **Validated on combined Joint Val Score** (.5 \times (\text{Entity F1} + \text{Linking F1})$).
   - *Each phase terminates dynamically if validation fails to improve for 3 consecutive epochs, restoring the best state dict before transitioning to the next phase.*
2. **Pairwise Sigmoid Focal Loss ($\gamma=2.0, \alpha=0.75$):** Eliminates gradient swamping from the 98.5% negative candidate pairs in (N^2)$ space.
3. **12-D Spatial Geometric Bias:** Strictly signed $\Delta x_{\text{gap}}, \Delta y_{\text{gap}}$, 1D vertical line overlap ($\text{IoU}_y$), and sequence reading-order step deltas.
4. **Soft Marginal Masking:** Replaces brittle Boolean $-\infty$ masking with entity posterior log-prior offsets.

> **Hardware**: Kaggle GPU (Tesla P100 16 GB or 2×T4). Expected runtime: ~15–20 minutes for 3 seeds with dynamic early stopping.

## 0 · Environment Setup & Git Clone

In [ ]:
# ── Clone or Pull KeyValue-Extraction Repository ─────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/ABasakEE/KeyValue-Extraction.git"
REPO_DIR = "/kaggle/working/KeyValue-Extraction"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"{REPO_DIR} already exists, pulling latest main branch...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Add src/ to sys.path
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print(f"sys.path: {sys.path[:2]}")

In [ ]:
# ── Install Required Dependencies ───────────────────────────────────────
!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval faker augraphy

In [ ]:
# ── Verify GPU and Environment ──────────────────────────────────────────
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Fine-tuning on CPU is prohibitively slow.")

## 1 · Download Raw FUNSD Dataset

In [ ]:
# ── Download Raw FUNSD (with full key-value linking annotations) ────────
FUNSD_DIR = os.path.join(REPO_DIR, "data", "funsd", "dataset")

if not os.path.isdir(FUNSD_DIR):
    print("Downloading official raw FUNSD release...")
    !mkdir -p data/funsd
    !wget -q https://guillaumejaume.github.io/FUNSD/dataset.zip -O data/funsd/dataset.zip
    !cd data/funsd && unzip -q dataset.zip && rm dataset.zip
    print("FUNSD downloaded and extracted successfully.")
else:
    print(f"FUNSD already present at {FUNSD_DIR}")

# Verify integrity of dataset paths
for split in ("training_data", "testing_data"):
    for sub in ("annotations", "images"):
        p = os.path.join(FUNSD_DIR, split, sub)
        n = len(os.listdir(p)) if os.path.isdir(p) else 0
        print(f"  {split}/{sub}: {n} files")

## 2 · Smoke Test Architecture Upgrades

Run unit tests verifying 12-D geometric spatial features, Soft Marginal Log-Priors, $K$-NN candidate masks, and Focal Loss.

In [ ]:
!python tests/test_linking_head.py


---
## 3 · Train Upgraded Joint Extractor on FUNSD

Trains LiLT with Decoupled 3-Stage Curriculum, Sigmoid Focal Loss, and 12-D Spatial Features across seeds (13, 21, 42).

In [ ]:
!python src/train.py --config configs/funsd_joint_lilt_curriculum.yaml

## 4 · Inspect Evaluation Metrics

In [ ]:
import json
from pathlib import Path

metrics_file = Path("results/funsd_joint_lilt_curriculum/metrics.json")
if metrics_file.exists():
    res = json.loads(metrics_file.read_text(encoding="utf-8"))
    print("=" * 65)
    print("UPGRADED TRACK A+ (CURRICULUM + FOCAL LOSS) RESULTS")
    print("=" * 65)
    print(f"Mean Entity F1 : {res.get('mean_f1', 0):.4f} +/- {res.get('std_f1', 0):.4f}")
    print(f"Mean Linking F1: {res.get('mean_linking_f1', 0):.4f} +/- {res.get('std_linking_f1', 0):.4f}")
    print("-" * 65)
    for run in res.get("runs", []):
        seed = run.get("seed", "?")
        ent_f1 = run.get("test", {}).get("f1", 0)
        lnk_f1 = run.get("test_linking", {}).get("f1", 0)
        print(f"  Seed {seed:>2} -> Entity F1: {ent_f1:.4f} | Linking F1: {lnk_f1:.4f}")
else:
    print(f"[-] Metrics file not found at {metrics_file}")

## 5 · Package Outputs for Local Download

Packs `metrics.json`, `config.json`, and the best checkpoint `.pt` into a single downloadable zip file.

In [ ]:
# Find best model across seeds and copy to root of output directory
import shutil
from pathlib import Path

out_dir = Path("results/funsd_joint_lilt_curriculum")
best_seed_model = None
best_f1 = -1.0

if metrics_file.exists():
    res = json.loads(metrics_file.read_text(encoding="utf-8"))
    for run in res.get("runs", []):
        seed = run.get("seed")
        score = run.get("val_score", run.get("test", {}).get("f1", 0))
        cand = out_dir / f"seed_{seed}" / "best_model.pt"
        if cand.exists() and score > best_f1:
            best_f1 = score
            best_seed_model = cand

if best_seed_model and best_seed_model.exists():
    target = out_dir / "best_model.pt"
    shutil.copy2(best_seed_model, target)
    print(f"Copied best checkpoint ({best_seed_model.parent.name}) to {target}")

# Create zip file for download in /kaggle/working
!cd /kaggle/working && zip -q -r /kaggle/working/kaggle_run3_results.zip KeyValue-Extraction/results/funsd_joint_lilt_curriculum -x "*.pyc" "*__pycache__*"
print("[+] Packaging complete: /kaggle/working/kaggle_run3_results.zip")
!ls -lh /kaggle/working/kaggle_run3_results.zip